# DPO: concise professional email rewriting

This notebook is a **thin demo**: every step calls into the tested `geap_tuning`
package. It mirrors [`examples/run_preference_email.py`](../examples/run_preference_email.py).

Where our other DPO demo tunes support-reply *warmth*, this teaches **concise +
professional** email rewriting and records an **honest before → after** lift with
two disciplines borrowed from the RLFT demo:

1. A **pilot gate** — score the untuned base's objective `mean_compression`; if the
   base already compresses aggressively there is no headroom, so don't tune.
2. A **head-to-head** before → after. The **headline is objective concision** (base
   vs tuned `mean_compression` + a `compression_win_rate` with a `bootstrap_ci`) —
   the exact axis the preference pairs train. A blind A/B **subjective** judge
   win-rate rides along as a secondary signal; a strong base can saturate it (ours
   already out-writes our gold on the judge while *expanding* drafts), so it can
   stay flat even as concision clearly improves — an honest lesson in what DPO moves.

> **Requires live GCP and incurs tuning cost** (one preference-tuning job). Have
> a real `.env` and `gcloud auth` in place.

In [ ]:
from geap_tuning.config import genai_client, load_config

BASE_MODEL = "gemini-2.5-flash"
JUDGE_MODEL = "gemini-2.5-flash"
MIN_BASE_COMPRESSION = 0.9  # base rewrite/draft word ratio must be >= this to have headroom
cfg = load_config()
client = genai_client(cfg)
cfg

## 1. Build the preference dataset and stage it to GCS

Each record is a `(draft, preferred, dispreferred)` triple — hand-authored so the
preferred rewrite is professional **and** materially shorter than the dispreferred
one (a concision invariant the unit tests enforce). The bank is 60 triples so the
0.25 test split gives ~15 held-out drafts — enough for a meaningful win-rate CI.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.preference.email import (
    EMAIL_DRAFTS,
    SYSTEM_INSTRUCTION,
    build_preference_dataset,
    build_preference_records,
    split_dataset,
)

GCS_PREFIX = "preference_concise_email_v2"
paths = build_preference_dataset("../datasets/preference_concise_email")
train_uri = upload_file(paths["train"], f"{cfg.bucket}/{GCS_PREFIX}/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/{GCS_PREFIX}/val.jsonl")

_, _, test_triples = split_dataset(EMAIL_DRAFTS)
test_records = build_preference_records(test_triples)
print(f"{len(EMAIL_DRAFTS)} triples, {len(test_records)} held out")

## 2. The blind A/B judge and the base generator

The judge prompt is **different** from the generator's `SYSTEM_INSTRUCTION` and
reads only the first letter of the verdict. The eval helpers randomize which slot
(A or B) each candidate lands in, so the judge's position bias cancels out.

In [ ]:
from geap_tuning.inference import generate

_JUDGE_PROMPT = (
    "You are judging two versions of the same work email. Pick the version that "
    "is more professional and concise (clear, brief, free of filler and hedging) "
    "while keeping the same information. Answer with only the single letter 'A' "
    "or 'B'.\n\n"
    "Original draft: {user}\n\nEmail A: {a}\n\nEmail B: {b}\n\nBetter email:"
)


def judge_fn(draft: str, cand_a: str, cand_b: str) -> str:
    verdict = generate(client, JUDGE_MODEL, _JUDGE_PROMPT.format(user=draft, a=cand_a, b=cand_b))
    return verdict[:1].upper()


def base_rewrite(draft: str) -> str:
    return generate(client, BASE_MODEL, draft, system_instruction=SYSTEM_INSTRUCTION)

## 3. Pilot gate — objective concision headroom (the "before")

`run_pilot_eval` scores the base's own `mean_compression` (rewrite/draft word
ratio). A ratio **at/above** `MIN_BASE_COMPRESSION` means the base barely shortens
(or even expands) the draft — real concision headroom for DPO to teach. It also
reports the subjective base-vs-gold win-rate for context: a strong base can already
out-write our hand-authored gold on the judge while still being verbose.

In [ ]:
from geap_tuning.preference.email_eval import run_pilot_eval

pilot = run_pilot_eval(test_records, base_rewrite, judge_fn)
print(
    f"BASE mean_compression={pilot['mean_compression']:.2f} (floor {MIN_BASE_COMPRESSION}); "
    f"subjective base-vs-gold win_rate={pilot['win_rate']:.3f} (context) (n={pilot['n']})"
)
if pilot["mean_compression"] >= MIN_BASE_COMPRESSION:
    print(
        "Pilot gate PASSED — the base barely shortens (or expands) the draft; headroom confirmed."
    )
else:
    print("WARNING: base already compresses aggressively (no headroom) — DPO may not show a lift.")

## 4. Launch the preference-tuning job and wait

A fresh display name (`-v2`) and a firmer pull toward the preferred (shorter)
completion than the defaults: `epochs=3`, `beta=0.2`.

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, tuned_endpoint, wait_for_tuning_job
from geap_tuning.preference.tune import launch_preference_job

DISPLAY_NAME = "geap-dpo-concise-email-v2"

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_preference_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        epochs=3,
        beta=0.2,
        labels=cfg.labels,
    )
job = wait_for_tuning_job(client, job.name)
endpoint = tuned_endpoint(job)


def tuned_rewrite(draft: str) -> str:
    return generate(client, endpoint, draft, system_instruction=SYSTEM_INSTRUCTION)


endpoint

## 5. Head-to-head before → after and report the lift

The **headline is objective concision**: base vs tuned `mean_compression` and a
`compression_win_rate` (fraction of drafts where the tuned rewrite is strictly
shorter than the base rewrite) with a `bootstrap_ci` — the exact axis the
preference pairs train. The subjective judge `win_rate` rides along as a secondary
signal; a strong base can hold it flat even as concision clearly improves.

In [ ]:
from geap_tuning.preference.email_eval import run_head_to_head_eval
from geap_tuning.rlft.evaluate import bootstrap_ci

h2h = run_head_to_head_eval(test_records, base_rewrite, tuned_rewrite, judge_fn)
low, high = bootstrap_ci(int(h2h["compression_hits"]), int(h2h["n"]))
print(
    f"HEADLINE (objective concision): mean_compression "
    f"base={h2h['base_mean_compression']:.2f} -> tuned={h2h['tuned_mean_compression']:.2f} "
    f"(lower is more concise)"
)
print(
    f"tuned shorter than base in {int(h2h['compression_hits'])}/{int(h2h['n'])} "
    f"(compression_win_rate={h2h['compression_win_rate']:.3f} CI[{low:.3f}, {high:.3f}])"
)
print(
    f"SECONDARY (subjective judge): tuned-vs-base win_rate={h2h['win_rate']:.3f} "
    "(a strong base can hold this flat even as concision improves)"
)